# HCE vs CE Baseline Comparison — v6 Data

This notebook trains two models on identical data and splits, differing only in their loss function:

| | **CE Baseline** | **HCE Model** |
|---|---|---|
| Architecture | C2S-Pythia-410m + linear head | Identical |
| Loss | Weighted cross-entropy (leaf classes only) | Hierarchical cross-entropy (paper Eq. 7) |
| Hierarchy knowledge | None | Full ontology DAG via reachability matrix |
| Class weights | Inverse frequency (same) | Inverse frequency (same) |
| Data | 8 organs (v6) | Identical |

**Hypothesis**: HCE improves macro recall on rare/coarse classes by propagating probability mass from fine-grained subtypes to their ancestors, reducing the number of classes with 0% recall.

**Key metrics**:
- In-distribution: accuracy, macro recall, macro F1, # classes with 0% recall
- Zero-shot: prediction breakdown on lab validation datasets

## 1. Imports

In [ ]:
import os, sys, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

import scanpy as sc
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.sparse as sp
import h5py

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
from cell2sentence.hce_trainer import build_reachability_matrix_from_ontology
from cell2sentence.hierarchy_utils import deduplicate_hierarchy

warnings.filterwarnings('ignore')
print('=' * 60)
print('  IMPORTS OK')
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}')
print('=' * 60)

## 2. Configuration

In [ ]:
# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR          = 'multi_tissue_v6_comparison_results'
HCE_MODEL_PATH   = os.path.join(OUT_DIR, 'hce_best_model.pt')
CE_MODEL_PATH    = os.path.join(OUT_DIR, 'ce_best_model.pt')
C2S_MODEL_NAME   = 'vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation'

# ── Identical to v6 ───────────────────────────────────────────────────────────
ORGAN_CONFIGS = [
    {'name': 'lung',        'path': 'lung.h5ad',                        'label_col': 'ann_finest_level', 'gene_col': 'feature_name', 'hierarchy_cols': ['ann_level_1','ann_level_2','ann_level_3','ann_level_4','ann_level_5'], 'coarse_col': None, 'id': 0},
    {'name': 'brain_glia',  'path': 'brain_new.h5ad',                  'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [], 'coarse_col': 'supercluster_term', 'id': 1},
    {'name': 'brain_neurons','path': 'brain_neurons_processed.h5ad',   'label_col': 'label',            'gene_col': 'feature_name', 'hierarchy_cols': [], 'coarse_col': None, 'id': 2},
    {'name': 'liver',       'path': 'census_data/liver.h5ad',           'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [], 'coarse_col': None, 'id': 3},
    {'name': 'lymph_node',  'path': 'census_data/lymph_node.h5ad',      'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [], 'coarse_col': None, 'id': 4},
    {'name': 'bone_marrow', 'path': 'census_data/bone_marrow.h5ad',     'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [], 'coarse_col': None, 'id': 5},
    {'name': 'lymphoid',    'path': 'lab-data/lymphoid.h5ad',            'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [], 'coarse_col': None, 'id': 6},
    {'name': 'myeloid',     'path': 'lab-data/myeloid.h5ad',             'label_col': 'cell_type',        'gene_col': 'feature_name', 'hierarchy_cols': [], 'coarse_col': None, 'id': 7},
]

LAB_CONFIGS = [
    {'name': 'All_cells (glioma)', 'path': 'All_cells.h5ad',                  'label_col': 'predicted.high_hierarchy'},
    {'name': 'Brain_normal',       'path': 'lab-data/Brain_normal.h5ad',       'label_col': 'cell_type'},
    {'name': 'Liver_normal',       'path': 'lab-data/Liver_normal.h5ad',       'label_col': 'cell_type'},
    {'name': 'Lymph_node_normal',  'path': 'lab-data/Lymph_node_normal.h5ad',  'label_col': 'cell_type'},
]

TOP_K_GENES        = 200
MAX_CELLS_PER_TYPE = 300
MIN_CELLS_PER_TYPE = 100
TEST_FRAC          = 0.15
VAL_FRAC           = 0.10
BATCH_SIZE         = 16
N_EPOCHS           = 10
LEARNING_RATE      = 1e-4
WEIGHT_DECAY       = 1e-2
WARMUP_STEPS       = 200
MAX_SEQ_LEN        = 512
MAX_WEIGHT         = 10.0
SEED               = 42

os.makedirs(OUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'  Device : {device}  |  Output dir : {OUT_DIR}')
print('[OK] Config ready')

## 3. Data Loading & Preprocessing
*(Identical to v6 — shared for both models)*

In [ ]:
def is_valid(value):
    if pd.isna(value): return False
    return str(value).strip().lower() not in ('', 'nan', 'none', 'unknown', 'na', 'n/a')

def get_gene_symbols(adata):
    if 'feature_name' in adata.var.columns:
        return np.array(adata.var['feature_name'].astype(str))
    return np.array(adata.var_names.astype(str))

def load_organ(cfg, min_cells=MIN_CELLS_PER_TYPE, max_cells=MAX_CELLS_PER_TYPE):
    name, label_col = cfg['name'], cfg['label_col']
    t0 = time.time()
    adata = sc.read_h5ad(cfg['path'], backed='r')
    gene_symbols = get_gene_symbols(adata)
    valid_mask = adata.obs[label_col].apply(is_valid)
    obs        = adata.obs[valid_mask].copy()
    valid_idx  = np.where(valid_mask.values)[0]
    labels  = obs[label_col].astype(str).values
    sampled = []
    for lbl in np.unique(labels):
        pos = np.where(labels == lbl)[0]
        if len(pos) > max_cells:
            pos = np.random.choice(pos, max_cells, replace=False)
        sampled.extend(pos.tolist())
    sampled   = np.sort(np.array(sampled, dtype=np.int64))
    obs       = obs.iloc[sampled].copy()
    valid_idx = valid_idx[sampled]
    counts  = obs[label_col].value_counts()
    keep    = counts[counts >= min_cells].index
    dropped = counts[counts < min_cells]
    if len(dropped):
        print(f'  [{name}] Dropping {len(dropped)} type(s) < {min_cells} cells')
    mask      = obs[label_col].isin(keep)
    obs       = obs[mask].copy()
    valid_idx = valid_idx[mask.values]
    print(f'  [{name}] {len(obs):,} cells | {obs[label_col].nunique()} types | {time.time()-t0:.1f}s')
    return adata, obs, valid_idx, gene_symbols

def cell_to_text_backed(h5_path, row_indices, gene_symbols, top_k=200, desc='cells'):
    texts = []
    with h5py.File(h5_path, 'r') as f:
        X = f['X']
        indptr = X['indptr'][:]
        indices_ds, data_ds = X['indices'], X['data']
        for row_idx in tqdm(row_indices, desc=f'  {desc}', leave=False):
            start, end = int(indptr[row_idx]), int(indptr[row_idx + 1])
            if start == end:
                texts.append('')
                continue
            vals = data_ds[start:end]
            cols = indices_ds[start:end]
            if len(vals) <= top_k:
                order = np.argsort(vals)[::-1]
            else:
                order = np.argpartition(vals, -top_k)[-top_k:]
                order = order[np.argsort(vals[order])[::-1]]
            texts.append(' '.join(str(gene_symbols[cols[j]]) for j in order if vals[j] > 0))
    return texts

def cell_to_text_dense(X_row, gene_symbols, top_k=200):
    row = X_row.toarray().flatten() if sp.issparse(X_row) else np.array(X_row).flatten()
    nz = np.where(row > 0)[0]
    if len(nz) == 0: return ''
    vals = row[nz]
    if len(nz) > top_k:
        idx = np.argpartition(vals, -top_k)[-top_k:]
        nz = nz[idx[np.argsort(vals[idx])[::-1]]]
    else:
        nz = nz[np.argsort(vals)[::-1]]
    return ' '.join(gene_symbols[i] for i in nz)

def cell_to_text_robust(h5_path, row_indices, gene_symbols, top_k=200, desc='cells'):
    try:
        return cell_to_text_backed(h5_path, row_indices, gene_symbols, top_k, desc)
    except (KeyError, AttributeError, TypeError, OSError):
        print(f'  [{desc}] CSR streaming failed — loading X into memory')
        adata_tmp = sc.read_h5ad(h5_path)
        X = adata_tmp.X
        texts = [cell_to_text_dense(X[i], gene_symbols, top_k)
                 for i in tqdm(row_indices, desc=f'  {desc}', leave=False)]
        del adata_tmp
        return texts

print('[OK] Utility functions defined')

In [ ]:
print('Loading all organs ...')
loaded_organs = {}
for cfg in ORGAN_CONFIGS:
    adata, obs, valid_idx, gene_syms = load_organ(cfg)
    loaded_organs[cfg['name']] = {'cfg': cfg, 'adata': adata, 'obs': obs,
                                   'valid_idx': valid_idx, 'gene_symbols': gene_syms}

# ── Label normalization: disease filter ───────────────────────────────────────
for organ_name in ['liver', 'lymph_node', 'bone_marrow', 'lymphoid', 'myeloid']:
    d = loaded_organs[organ_name]
    obs = d['obs']
    if 'disease' in obs.columns:
        non_normal = (obs['disease'] != 'normal').sum()
        if non_normal:
            mask = obs['disease'] == 'normal'
            d['obs'] = obs[mask].copy()
            d['valid_idx'] = d['valid_idx'][mask.values]
            print(f'  [{organ_name}] Removed {non_normal} non-normal cells')

# Fix lung pericyte naming
loaded_organs['lung']['obs']['ann_finest_level'] = (
    loaded_organs['lung']['obs']['ann_finest_level'].replace('Pericytes', 'pericyte')
)

print('[OK] All organs loaded and normalized')

In [ ]:
LABEL_SYNONYM_MAP = {
    'B cells': 'B cell', 'NK cells': 'natural killer cell',
    'Alveolar macrophages': 'alveolar macrophage', 'Mast cells': 'mast cell',
    'Plasma cells': 'plasma cell', 'Classical monocytes': 'classical monocyte',
    'Non-classical monocytes': 'non-classical monocyte',
    'Plasmacytoid DCs': 'plasmacytoid dendritic cell',
    'Smooth muscle': 'smooth muscle cell',
    'CD4 T cells': 'CD4-positive, alpha-beta T cell',
    'CD4-positive helper T cell': 'CD4-positive, alpha-beta T cell',
    'CD8 T cells': 'CD8-positive, alpha-beta T cell',
    'CD4-positive, CD25-positive, alpha-beta regulatory T cell': 'regulatory T cell',
    'CD8-positive, alpha-beta memory T cell, CD45RO-positive': 'CD8-positive, alpha-beta memory T cell',
    'mature alpha-beta T cell': 'alpha-beta T cell',
    'mature NK T cell': 'natural killer T cell',
    'mature B cell': 'B cell',
    'effector memory CD4-positive, alpha-beta T cell, terminally differentiated': 'effector memory CD4-positive, alpha-beta T cell',
    'dendritic cell, human': 'dendritic cell',
    'group 3 innate lymphoid cell, human': 'group 3 innate lymphoid cell',
    'plasmacytoid dendritic cell, human': 'plasmacytoid dendritic cell',
    'CD14-positive monocyte': 'classical monocyte',
    'CD14-positive, CD16-positive monocyte': 'intermediate monocyte',
    'myeloid dendritic cell': 'conventional dendritic cell',
    'liver dendritic cell': 'conventional dendritic cell',
    'vein endothelial cell': 'endothelial cell of vein',
    'endothelial cell of pericentral hepatic sinusoid': 'endothelial cell of hepatic sinusoid',
    'endothelial cell of periportal hepatic sinusoid': 'endothelial cell of hepatic sinusoid',
    'intrahepatic cholangiocyte': 'cholangiocyte',
    'granulocyte monocyte progenitor cell': 'granulocyte monocyte progenitor',
    'cycling plasma cell': 'plasma cell',
    'inflammatory macrophage': 'macrophage',
    'B_cell': 'B cell', 'B_cell_naive': 'naive B cell',
    'NK_cell': 'natural killer cell', 'plasma_cell': 'plasma cell',
    'plasma_cell_proliferating': 'plasma cell', 'Treg_cell': 'regulatory T cell',
    'alveolar_macrophage': 'alveolar macrophage', 'mast_cell': 'mast cell',
    'monocyte_CSF3R': 'classical monocyte', 'monocyte_SOCS3': 'classical monocyte',
    'monocyte_ITGAL': 'non-classical monocyte', 'monocyte_AREG_EREG': 'monocyte',
    'cDC1': 'conventional dendritic cell',
}

# Bad label filters
BAD_LABEL_FILTERS = [
    ('liver', 'cell_type', {'malignant cell'}),
    ('lymph_node', 'cell_type', {'stromal cell of pancreas', 'alveolar macrophage'}),
]
for organ_name, col, bad_labels in BAD_LABEL_FILTERS:
    d, obs = loaded_organs[organ_name], loaded_organs[organ_name]['obs']
    for lbl in bad_labels:
        n = (obs[col] == lbl).sum()
        if n:
            mask = obs[col] != lbl
            d['obs'] = obs[mask].copy()
            d['valid_idx'] = d['valid_idx'][mask.values]
            obs = d['obs']
            print(f'  [{organ_name}] Dropped "{lbl}" ({n} cells)')

# Apply synonyms
total_remapped = 0
for cfg in ORGAN_CONFIGS:
    name, col = cfg['name'], cfg['label_col']
    obs = loaded_organs[name]['obs']
    for src, tgt in LABEL_SYNONYM_MAP.items():
        n = (obs[col] == src).sum()
        if n:
            loaded_organs[name]['obs'][col] = obs[col].replace(src, tgt)
            obs = loaded_organs[name]['obs']
            total_remapped += n
print(f'[OK] Synonyms applied — {total_remapped:,} cells remapped')

In [ ]:
# ── Lung hierarchy ────────────────────────────────────────────────────────────
lung_cfg  = loaded_organs['lung']['cfg']
lung_obs2 = loaded_organs['lung']['obs']
lung_ontology = {}
level_cols = [c for c in lung_cfg['hierarchy_cols'] if c in lung_obs2.columns]
cols_ordered = level_cols + [lung_cfg['label_col']]
for k in range(1, len(cols_ordered)):
    parent_col, child_col = cols_ordered[k-1], cols_ordered[k]
    for _, row in lung_obs2[[parent_col, child_col]].dropna().drop_duplicates().iterrows():
        p, ch = str(row[parent_col]), str(row[child_col])
        if is_valid(p) and is_valid(ch) and ch not in lung_ontology:
            lung_ontology[ch] = p
for val in lung_obs2[cols_ordered[0]].dropna().unique():
    if is_valid(val) and str(val) not in lung_ontology:
        lung_ontology[str(val)] = None

# ── Brain glia hierarchy ──────────────────────────────────────────────────────
brain_glia_cfg = loaded_organs['brain_glia']['cfg']
brain_glia_obs = loaded_organs['brain_glia']['obs']
brain_glia_ontology = {}
coarse_col = brain_glia_cfg['coarse_col']
for _, row in brain_glia_obs[[coarse_col, brain_glia_cfg['label_col']]].dropna().drop_duplicates().iterrows():
    brain_glia_ontology[str(row[brain_glia_cfg['label_col']])] = str(row[coarse_col])
for val in brain_glia_obs[coarse_col].dropna().unique():
    if str(val) not in brain_glia_ontology:
        brain_glia_ontology[str(val)] = None

# ── Cross-organ hierarchy (v6) ────────────────────────────────────────────────
CROSS_ORGAN_HIERARCHY = {
    'Upper-layer intratelencephalic': 'excitatory neuron', 'Deep-layer intratelencephalic': 'excitatory neuron',
    'Deep-layer corticothalamic and 6b': 'excitatory neuron', 'Deep-layer near-projecting': 'excitatory neuron',
    'Hippocampal CA1-3': 'excitatory neuron', 'Hippocampal CA4': 'excitatory neuron',
    'Hippocampal dentate gyrus': 'excitatory neuron', 'Thalamic excitatory': 'excitatory neuron',
    'Amygdala excitatory': 'excitatory neuron', 'Upper rhombic lip': 'excitatory neuron',
    'Lower rhombic lip': 'excitatory neuron', 'Mammillary body': 'excitatory neuron',
    'CGE interneuron': 'inhibitory neuron', 'MGE interneuron': 'inhibitory neuron',
    'Cerebellar inhibitory': 'inhibitory neuron', 'LAMP5-LHX6 and Chandelier': 'inhibitory neuron',
    'Medium spiny neuron': 'inhibitory neuron', 'Eccentric medium spiny neuron': 'inhibitory neuron',
    'Midbrain-derived inhibitory': 'inhibitory neuron', 'Miscellaneous': 'neuron',
    'excitatory neuron': 'neuron', 'inhibitory neuron': 'neuron',
    'CD4-positive, alpha-beta T cell': 'alpha-beta T cell', 'CD8-positive, alpha-beta T cell': 'alpha-beta T cell',
    'regulatory T cell': 'CD4-positive, alpha-beta T cell', 'T follicular helper cell': 'CD4-positive, alpha-beta T cell',
    'naive thymus-derived CD4-positive, alpha-beta T cell': 'CD4-positive, alpha-beta T cell',
    'central memory CD4-positive, alpha-beta T cell': 'CD4-positive, alpha-beta T cell',
    'effector memory CD4-positive, alpha-beta T cell': 'CD4-positive, alpha-beta T cell',
    'naive thymus-derived CD8-positive, alpha-beta T cell': 'CD8-positive, alpha-beta T cell',
    'central memory CD8-positive, alpha-beta T cell': 'CD8-positive, alpha-beta T cell',
    'effector memory CD8-positive, alpha-beta T cell': 'CD8-positive, alpha-beta T cell',
    'effector CD8-positive, alpha-beta T cell': 'CD8-positive, alpha-beta T cell',
    'gamma-delta T cell': 'T cell', 'mucosal invariant T cell': 'T cell',
    'natural killer T cell': 'T cell', 'alpha-beta T cell': 'T cell', 'T cell': 'lymphocyte',
    'naive B cell': 'B cell', 'memory B cell': 'B cell', 'germinal center B cell': 'B cell',
    'plasmablast': 'B cell', 'plasma cell': 'B cell', 'transitional stage B cell': 'B cell',
    'B cell': 'lymphocyte', 'natural killer cell': 'lymphocyte', 'innate lymphoid cell': 'lymphocyte',
    'group 1 innate lymphoid cell': 'innate lymphoid cell', 'group 2 innate lymphoid cell': 'innate lymphoid cell',
    'group 3 innate lymphoid cell': 'innate lymphoid cell', 'lymphocyte': 'leukocyte',
    'classical monocyte': 'monocyte', 'non-classical monocyte': 'monocyte',
    'intermediate monocyte': 'monocyte', 'monocyte': 'myeloid leukocyte',
    'Kupffer cell': 'macrophage', 'alveolar macrophage': 'macrophage', 'macrophage': 'myeloid leukocyte',
    'conventional dendritic cell': 'dendritic cell', 'plasmacytoid dendritic cell': 'dendritic cell',
    'dendritic cell': 'myeloid leukocyte', 'mast cell': 'myeloid leukocyte',
    'neutrophil': 'myeloid leukocyte', 'basophil': 'myeloid leukocyte',
    'eosinophil': 'myeloid leukocyte', 'myeloid leukocyte': 'leukocyte', 'leukocyte': 'Immune',
    'hematopoietic stem cell': 'hematopoietic precursor cell',
    'common myeloid progenitor': 'hematopoietic precursor cell',
    'granulocyte monocyte progenitor': 'hematopoietic precursor cell',
    'common lymphoid progenitor': 'hematopoietic precursor cell',
    'hematopoietic precursor cell': 'Immune',
    'proerythroblast': 'erythroid lineage cell', 'erythroblast': 'erythroid lineage cell',
    'reticulocyte': 'erythroid lineage cell', 'erythrocyte': 'erythroid lineage cell',
    'erythroid lineage cell': 'hematopoietic precursor cell',
    'megakaryocyte-erythroid progenitor cell': 'hematopoietic precursor cell',
    'megakaryocyte': 'hematopoietic precursor cell', 'platelet': 'megakaryocyte',
    'endothelial cell of artery': 'endothelial cell', 'endothelial cell of vein': 'endothelial cell',
    'endothelial cell of hepatic sinusoid': 'endothelial cell',
    'blood vessel endothelial cell': 'endothelial cell', 'lymphatic endothelial cell': 'endothelial cell',
    'high endothelial venule cell': 'endothelial cell', 'capillary endothelial cell': 'endothelial cell',
    'hepatic stellate cell': 'fibroblast', 'portal fibroblast': 'fibroblast',
    'fibroblastic reticular cell': 'fibroblast',
    'hepatocyte': 'hepatic cell', 'cholangiocyte': 'hepatic cell', 'hepatic cell': 'Epithelial',
    'Oligodendrocyte': 'glial cell', 'Astrocyte': 'glial cell', 'Microglia': 'glial cell',
    'Committed oligodendrocyte precursor': 'glial cell', 'Oligodendrocyte precursor': 'glial cell',
    'Bergmann glia': 'glial cell', 'Choroid plexus': 'glial cell', 'Ependymal': 'glial cell',
    'Fibroblast': 'Fibroblast lineage',
    # Step 1: wired atlas labels
    'Interstitial Mph perivascular': 'macrophage', 'Peribronchial fibroblasts': 'fibroblast',
    'EC general capillary': 'capillary endothelial cell', 'EC venous pulmonary': 'endothelial cell of vein',
    'Lymphatic EC mature': 'lymphatic endothelial cell',
    'central nervous system macrophage': 'macrophage', 'microglial cell': 'macrophage',
    # Step 2a: lymphoid activation state leaves
    'CD4_T_cell_activated': 'CD4-positive, alpha-beta T cell',
    'CD4_T_cell_naive_or_memory': 'CD4-positive, alpha-beta T cell',
    'CD8_T_cell_early_activated': 'CD8-positive, alpha-beta T cell',
    'CD8_T_cell_late_exhausted': 'CD8-positive, alpha-beta T cell',
    'CD8_T_cell_proliferating': 'CD8-positive, alpha-beta T cell',
    # Step 2b: myeloid macrophage functional state leaves
    'macrophage_APOE_CHIT': 'macrophage', 'macrophage_C3': 'macrophage',
    'macrophage_F13A1': 'macrophage', 'macrophage_FOLR2': 'macrophage',
    'macrophage_ISG_expressing': 'macrophage', 'macrophage_VEGFA': 'macrophage',
    'macrophage_glycolytic': 'macrophage',
    # Step 3: neuron fallback
    'neuron (unspecified)': 'neuron', 'GABAergic_interneuron': 'inhibitory neuron',
    'GABAergic_interneuron_SST': 'MGE interneuron', 'astrocyte_fibrous_like': 'Astrocyte',
    'astrocyte_protoplasmic': 'Astrocyte', 'arterial_endothelial_cell': 'endothelial cell of artery',
    'capillary_endothelial_cell': 'capillary endothelial cell',
    'venous_endothelial_cell': 'endothelial cell of vein',
    'meningeal_fibroblast': 'fibroblast', 'perivascular_fibroblast': 'fibroblast',
    'smooth_muscle_cell': 'smooth muscle cell', 'oligodendrocyte_ISG_expressing': 'Oligodendrocyte',
    'oligodendrocyte_precursor_cell': 'Oligodendrocyte precursor',
}

# Merge ontologies
combined_ontology = {}
combined_ontology.update(lung_ontology)
combined_ontology.update(brain_glia_ontology)
combined_ontology.update(CROSS_ORGAN_HIERARCHY)
for organ_name in ['liver', 'lymph_node', 'bone_marrow', 'lymphoid', 'myeloid']:
    d = loaded_organs[organ_name]
    for ct in d['obs'][d['cfg']['label_col']].unique():
        ct = str(ct)
        if ct not in combined_ontology:
            combined_ontology[ct] = None
combined_ontology['pericyte'] = 'Vascular'

# Deduplicate collisions
for cfg in ORGAN_CONFIGS:
    obs, combined_ontology, report = deduplicate_hierarchy(
        loaded_organs[cfg['name']]['obs'], combined_ontology, cfg['label_col']
    )
    loaded_organs[cfg['name']]['obs'] = obs
    if not report.empty:
        for _, row in report.iterrows():
            print(f'  [{cfg["name"]}] "{row["original_label"]}" -> "{row["new_label"]}" ({row["n_cells_renamed"]})')

# Ensure synonym targets are in ontology
for tgt in set(LABEL_SYNONYM_MAP.values()):
    if tgt not in combined_ontology:
        combined_ontology[tgt] = None

print(f'[OK] Combined ontology: {len(combined_ontology)} entries')

In [ ]:
print('Cell-to-text conversion ...')
CSR_ORGANS = {'lung', 'brain_glia', 'brain_neurons', 'liver', 'lymph_node', 'bone_marrow'}
all_texts, all_labels_str, all_organ_ids = [], [], []
organ_texts, organ_labels = {}, {}

for cfg in ORGAN_CONFIGS:
    name = cfg['name']
    d    = loaded_organs[name]
    print(f'  [{name}] {len(d["valid_idx"]):,} cells ...')
    texts = (cell_to_text_backed if name in CSR_ORGANS else cell_to_text_robust)(
        cfg['path'], d['valid_idx'], d['gene_symbols'], TOP_K_GENES, desc=name
    )
    keep_mask = [bool(t.strip()) for t in texts]
    texts     = [t for t, k in zip(texts, keep_mask) if k]
    labels    = d['obs'][cfg['label_col']].astype(str).values[keep_mask]
    organ_texts[name], organ_labels[name] = texts, labels
    all_texts.extend(texts)
    all_labels_str.extend(labels)
    all_organ_ids.extend([cfg['id']] * len(texts))

all_labels_str = np.array(all_labels_str)
all_organ_ids  = np.array(all_organ_ids, dtype=int)
print(f'[OK] {len(all_texts):,} cells | {len(np.unique(all_labels_str))} unique labels')

In [ ]:
print('Building vocabulary and splits ...')
tokenizer = AutoTokenizer.from_pretrained(C2S_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

leaf_classes_set = set(all_labels_str)
all_nodes = set()
for child, parent in combined_ontology.items():
    all_nodes.add(child)
    if parent: all_nodes.add(parent)
all_nodes |= leaf_classes_set

class_names   = sorted(all_nodes)
class_to_idx  = {name: idx for idx, name in enumerate(class_names)}
n_classes     = len(class_names)
leaf_classes  = sorted(leaf_classes_set)
leaf_indices  = [class_to_idx[c] for c in leaf_classes]
leaf_index_set = set(leaf_indices)

# Leaf-local index mapping (for CE baseline)
leaf_to_local   = {leaf_idx: i for i, leaf_idx in enumerate(leaf_indices)}
local_to_leaf   = {i: leaf_idx for i, leaf_idx in enumerate(leaf_indices)}
n_leaf_classes  = len(leaf_classes)

labels_encoded = np.array([class_to_idx[s] for s in all_labels_str], dtype=int)

class CellTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)}

strat_key = labels_encoded * 10 + all_organ_ids
idx_all   = np.arange(len(all_texts))
idx_tv, idx_test   = train_test_split(idx_all, test_size=TEST_FRAC, stratify=strat_key, random_state=SEED)
idx_train, idx_val = train_test_split(idx_tv,  test_size=VAL_FRAC/(1-TEST_FRAC), stratify=strat_key[idx_tv], random_state=SEED)

train_labels = labels_encoded[idx_train]
val_labels   = labels_encoded[idx_val]
test_labels  = labels_encoded[idx_test]
test_organ   = all_organ_ids[idx_test]

train_ds = CellTextDataset([all_texts[i] for i in idx_train], train_labels, tokenizer, MAX_SEQ_LEN)
val_ds   = CellTextDataset([all_texts[i] for i in idx_val],   val_labels,   tokenizer, MAX_SEQ_LEN)
test_ds  = CellTextDataset([all_texts[i] for i in idx_test],  test_labels,  tokenizer, MAX_SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'  Total vocab : {n_classes}  |  Leaf classes : {n_leaf_classes}')
print(f'  Train : {len(idx_train):,} | Val : {len(idx_val):,} | Test : {len(idx_test):,}')
print('[OK] Vocabulary and splits ready')

## 4. Loss Functions & Class Weights
*(Built once, shared by both models)*

In [ ]:
print('Building reachability matrix and class weights ...')

# ── Reachability matrix (HCE only) ────────────────────────────────────────────
R_np = build_reachability_matrix_from_ontology(combined_ontology, class_names)
reachability_matrix = torch.tensor(R_np, dtype=torch.float32).to(device)
print(f'  R shape : {n_classes}x{n_classes} | nnz={int(reachability_matrix.sum())}')

# ── Class weights: w_i = N / (C * n_i) ───────────────────────────────────────
train_counts   = Counter(train_labels.tolist())
N_train, C_obs = len(train_labels), len(train_counts)

# Full-vocab weights (for HCE — ancestors get effective-count weight)
class_weights_full = torch.zeros(n_classes, dtype=torch.float32, device=device)
for idx_ct, count in train_counts.items():
    class_weights_full[idx_ct] = N_train / (C_obs * count)
ancestor_indices = [i for i in range(n_classes) if i not in leaf_index_set]
for anc_idx in ancestor_indices:
    eff = sum(train_counts.get(j, 0) for j in leaf_indices if R_np[anc_idx, j] > 0)
    if eff > 0:
        class_weights_full[anc_idx] = N_train / (C_obs * eff)
class_weights_full = torch.clamp(class_weights_full, max=MAX_WEIGHT)

# Leaf-only weights (for CE baseline — same inverse-frequency weighting)
leaf_train_counts = {leaf_to_local[k]: v for k, v in train_counts.items() if k in leaf_to_local}
N_leaf, C_leaf = sum(leaf_train_counts.values()), len(leaf_train_counts)
class_weights_leaf = torch.zeros(n_leaf_classes, dtype=torch.float32, device=device)
for local_idx, count in leaf_train_counts.items():
    class_weights_leaf[local_idx] = N_leaf / (C_leaf * count)
class_weights_leaf = torch.clamp(class_weights_leaf, max=MAX_WEIGHT)

# ── HCE loss ──────────────────────────────────────────────────────────────────
class HCELoss(nn.Module):
    """Hierarchical cross-entropy: s = softmax(logits) @ R^T, loss = -w_t * log(s_t)."""
    def __init__(self, R, class_weights=None, eps=1e-8):
        super().__init__()
        self.register_buffer('R', R)
        self.eps = eps
        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)
        else:
            self.class_weights = None
    def forward(self, logits, targets):
        probs  = torch.softmax(logits, dim=1)
        s      = torch.clamp(probs @ self.R.T, min=self.eps)
        log_st = torch.log(s)[torch.arange(len(targets), device=targets.device), targets]
        if self.class_weights is not None:
            return -(self.class_weights[targets] * log_st).mean()
        return -log_st.mean()

# ── CE baseline loss ──────────────────────────────────────────────────────────
class CEBaselineLoss(nn.Module):
    """
    Standard weighted cross-entropy restricted to leaf classes.
    The model head outputs n_classes logits (same as HCE), but only the
    leaf-class columns are used during training — no hierarchy knowledge.
    """
    def __init__(self, leaf_indices, class_weights_leaf=None):
        super().__init__()
        self.register_buffer('leaf_idx', torch.tensor(leaf_indices, dtype=torch.long))
        if class_weights_leaf is not None:
            self.register_buffer('class_weights', class_weights_leaf)
        else:
            self.class_weights = None
    def forward(self, logits, targets):
        # Restrict to leaf columns — map global target indices to local leaf indices
        leaf_logits = logits[:, self.leaf_idx]  # (B, n_leaves)
        leaf_targets = torch.tensor(
            [leaf_to_local[t.item()] for t in targets],
            dtype=torch.long, device=targets.device
        )
        return F.cross_entropy(leaf_logits, leaf_targets,
                               weight=self.class_weights, reduction='mean')

hce_criterion = HCELoss(reachability_matrix, class_weights_full).to(device)
ce_criterion  = CEBaselineLoss(leaf_indices, class_weights_leaf).to(device)

print('[OK] Loss functions ready')
print(f'  HCE class weights : {(class_weights_full > 0).sum().item()}/{n_classes} non-zero')
print(f'  CE  class weights : {(class_weights_leaf > 0).sum().item()}/{n_leaf_classes} non-zero')

## 5. Model Architecture
*(Shared — both models use the same C2SClassifier)*

In [ ]:
class C2SClassifier(nn.Module):
    """C2S-Pythia-410m encoder + dropout + linear head over n_classes."""
    def __init__(self, encoder, hidden_size, num_classes):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.head    = nn.Linear(hidden_size, num_classes)
    def forward(self, input_ids, attention_mask):
        out         = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state
        seq_len     = attention_mask.sum(dim=1) - 1
        last_token  = last_hidden[torch.arange(last_hidden.size(0), device=last_hidden.device), seq_len]
        return self.head(self.dropout(last_token))

def build_fresh_model():
    """Load a fresh C2S encoder and wrap in C2SClassifier."""
    enc = AutoModel.from_pretrained(C2S_MODEL_NAME)
    enc.gradient_checkpointing_enable()
    return C2SClassifier(enc, enc.config.hidden_size, n_classes).to(device)

def build_optimizer_and_schedulers(model, n_steps):
    opt = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    warmup = optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=WARMUP_STEPS)
    cosine = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, n_steps - WARMUP_STEPS))
    return opt, warmup, cosine

total_params = sum(p.numel() for p in build_fresh_model().parameters()) / 1e6
print(f'[OK] Architecture: C2S-Pythia-410m + head  |  {total_params:.1f}M params  |  {n_classes} output classes')

## 6. Training Loop (Generic)
*(Same loop called twice — once per model)*

In [ ]:
leaf_indices_t = torch.tensor(leaf_indices, device=device)

def train_model(model, criterion, save_path, label):
    """
    Train model for N_EPOCHS, save best checkpoint by val accuracy.
    Returns history dict and best_val_acc.
    """
    total_steps = len(train_loader) * N_EPOCHS
    optimizer, warmup_sched, cosine_sched = build_optimizer_and_schedulers(model, total_steps)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, global_step = 0.0, 0

    print(f'\n{"="*60}')
    print(f'  Training [{label}]')
    print(f'  Steps/epoch: {len(train_loader)}  |  Total: {total_steps}')
    print('=' * 60)
    t_start = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        t_epoch = time.time()
        model.train()
        running_loss, n_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f'  [{label}] Epoch {epoch}/{N_EPOCHS} [train]', leave=True)
        for batch in pbar:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_b       = batch['label'].to(device)
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if global_step < WARMUP_STEPS: warmup_sched.step()
            else: cosine_sched.step()
            global_step += 1
            running_loss += loss.item()
            n_batches    += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')
        train_loss = running_loss / n_batches

        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f'  [{label}] Epoch {epoch}/{N_EPOCHS} [val]  ', leave=False):
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels_b       = batch['label'].to(device)
                logits         = model(input_ids, attention_mask)
                val_loss_sum  += criterion(logits, labels_b).item() * len(labels_b)
                preds          = leaf_indices_t[logits[:, leaf_indices_t].argmax(dim=1)]
                val_correct   += (preds == labels_b).sum().item()
                val_total     += len(labels_b)

        val_loss = val_loss_sum / val_total
        val_acc  = val_correct  / val_total
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_acc': val_acc, 'label': label}, save_path)

        print(f'  [{label}] Epoch {epoch}/{N_EPOCHS} | train={train_loss:.4f} | '
              f'val={val_loss:.4f} | val_acc={val_acc:.4f} | {"** BEST **" if improved else ""} | {time.time()-t_epoch:.0f}s')

    print(f'\n  [{label}] Total time : {(time.time()-t_start)/60:.1f} min  |  Best val acc : {best_val_acc:.4f}')
    return history, best_val_acc

print('[OK] Training loop defined')

## 7. Train HCE Model

In [ ]:
hce_model   = build_fresh_model()
hce_history, hce_best_acc = train_model(hce_model, hce_criterion, HCE_MODEL_PATH, 'HCE')

## 8. Train CE Baseline Model

In [ ]:
ce_model   = build_fresh_model()
ce_history, ce_best_acc = train_model(ce_model, ce_criterion, CE_MODEL_PATH, 'CE')

## 9. Evaluate Both Models
*(Load best checkpoints, run on test set)*

In [ ]:
def evaluate_model(model, ckpt_path, label):
    """
    Load best checkpoint, run inference on test set.
    Returns (preds_array, trues_array, per_class_df_per_organ).
    """
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'  [{label}] Loaded epoch {ckpt["epoch"]}, val_acc={ckpt["val_acc"]:.4f}')

    model.eval()
    preds_all, trues_all = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'  [{label}] Evaluating', leave=True):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds  = leaf_indices_t[logits[:, leaf_indices_t].argmax(dim=1)]
            preds_all.extend(preds.cpu().numpy())
            trues_all.extend(batch['label'].numpy())
    return np.array(preds_all), np.array(trues_all)


def compute_metrics(trues, preds, organ_id=None):
    """Return dict of metrics for a subset of test samples."""
    if organ_id is not None:
        mask  = test_organ == organ_id
        trues = trues[mask]
        preds = preds[mask]
    if len(trues) == 0:
        return None
    unique = np.unique(trues)
    acc    = accuracy_score(trues, preds)
    p, r, f, _ = precision_recall_fscore_support(trues, preds, labels=unique, average='macro', zero_division=0)
    p_pc, r_pc, f_pc, sup = precision_recall_fscore_support(trues, preds, labels=unique, zero_division=0)
    per_class = pd.DataFrame({
        'cell_type': [class_names[i] for i in unique],
        'precision': p_pc, 'recall': r_pc, 'f1': f_pc, 'support': sup,
    })
    return {
        'accuracy': acc, 'macro_recall': r, 'macro_f1': f, 'macro_precision': p,
        'zero_recall_count': int((per_class['recall'] == 0).sum()),
        'recall_50_pct': float((per_class['recall'] >= 0.5).mean()),
        'recall_80_pct': float((per_class['recall'] >= 0.8).mean()),
        'per_class': per_class,
        'n_samples': len(trues),
    }


print('Evaluating HCE model ...')
hce_preds, hce_trues = evaluate_model(hce_model, HCE_MODEL_PATH, 'HCE')

print('\nEvaluating CE baseline model ...')
ce_preds,  ce_trues  = evaluate_model(ce_model,  CE_MODEL_PATH,  'CE')

# Per-organ and combined metrics
results = {}  # 'hce' / 'ce' -> {organ_name -> metrics_dict}
for model_label, preds, trues in [('hce', hce_preds, hce_trues), ('ce', ce_preds, ce_trues)]:
    results[model_label] = {}
    for cfg in ORGAN_CONFIGS:
        m = compute_metrics(trues, preds, cfg['id'])
        if m:
            results[model_label][cfg['name']] = m
    results[model_label]['COMBINED'] = compute_metrics(trues, preds)

print('\n[OK] Evaluation complete')

## 10. Side-by-Side Comparison Table

In [ ]:
print('=' * 80)
print('  HCE vs CE BASELINE — TEST SET COMPARISON')
print('=' * 80)

metric_cols = ['accuracy', 'macro_recall', 'macro_f1', 'zero_recall_count',
               'recall_50_pct', 'recall_80_pct']

rows = []
for organ_name in [c['name'] for c in ORGAN_CONFIGS] + ['COMBINED']:
    hce_m = results['hce'].get(organ_name)
    ce_m  = results['ce'].get(organ_name)
    if hce_m is None or ce_m is None:
        continue
    row = {'organ': organ_name}
    for col in metric_cols:
        row[f'HCE_{col}'] = hce_m[col]
        row[f'CE_{col}']  = ce_m[col]
        row[f'delta_{col}'] = hce_m[col] - ce_m[col]
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index('organ')

# Pretty print
print(f'\n{"Organ":<22} {"Metric":<20} {"CE":>8} {"HCE":>8} {"Delta (HCE-CE)":>16}')
print('-' * 78)
for organ_name, row in comparison_df.iterrows():
    first = True
    for col in metric_cols:
        ce_val  = row[f'CE_{col}']
        hce_val = row[f'HCE_{col}']
        delta   = row[f'delta_{col}']
        sign    = '+' if delta > 0 else ''
        # For zero_recall_count, lower is better — flip sign display
        is_lower_better = col == 'zero_recall_count'
        better = (delta < 0) if is_lower_better else (delta > 0)
        indicator = ' <--' if abs(delta) > 0.01 or (is_lower_better and abs(delta) >= 1) else ''
        label = organ_name if first else ''
        fmt_hce = f'{hce_val:.0f}' if col == 'zero_recall_count' else f'{hce_val:.4f}'
        fmt_ce  = f'{ce_val:.0f}'  if col == 'zero_recall_count' else f'{ce_val:.4f}'
        fmt_d   = f'{delta:+.0f}' if col == 'zero_recall_count' else f'{delta:+.4f}'
        print(f'  {label:<20} {col:<20} {fmt_ce:>8} {fmt_hce:>8} {fmt_d:>16}{indicator}')
        first = False
    print()

comparison_df.to_csv(os.path.join(OUT_DIR, 'hce_vs_ce_comparison.csv'))
print(f'[OK] Comparison saved to {OUT_DIR}/hce_vs_ce_comparison.csv')

## 11. Training Curve Comparison

In [ ]:
epochs_ax = np.arange(1, N_EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Val accuracy
axes[0].plot(epochs_ax, hce_history['val_acc'], 'o-', color='steelblue', label='HCE')
axes[0].plot(epochs_ax, ce_history['val_acc'],  's-', color='tomato',    label='CE Baseline')
axes[0].set(xlabel='Epoch', ylabel='Val Accuracy', title='Validation Accuracy', ylim=[0, 1])
axes[0].legend(); axes[0].grid(alpha=.3)

# Val loss (note: different scale — CE and HCE losses are not comparable)
axes[1].plot(epochs_ax, hce_history['val_loss'], 'o-', color='steelblue', label='HCE')
axes[1].plot(epochs_ax, ce_history['val_loss'],  's-', color='tomato',    label='CE Baseline')
axes[1].set(xlabel='Epoch', ylabel='Val Loss', title='Validation Loss\n(CE and HCE losses not directly comparable)')
axes[1].legend(); axes[1].grid(alpha=.3)

# Train loss
axes[2].plot(epochs_ax, hce_history['train_loss'], 'o-', color='steelblue', label='HCE')
axes[2].plot(epochs_ax, ce_history['train_loss'],  's-', color='tomato',    label='CE Baseline')
axes[2].set(xlabel='Epoch', ylabel='Train Loss', title='Training Loss')
axes[2].legend(); axes[2].grid(alpha=.3)

plt.suptitle('HCE vs CE Baseline — Training Curves', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_curves_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12. Per-Class Recall Comparison
*(Head-to-head recall for every class — sorted by HCE recall)*

In [ ]:
def plot_recall_comparison(organ_name, hce_metrics, ce_metrics):
    hce_pc = hce_metrics['per_class'].set_index('cell_type')['recall']
    ce_pc  = ce_metrics['per_class'].set_index('cell_type')['recall']
    # Union of all classes
    all_ct = sorted(set(hce_pc.index) | set(ce_pc.index),
                    key=lambda ct: hce_pc.get(ct, 0))
    hce_vals = [hce_pc.get(ct, 0) for ct in all_ct]
    ce_vals  = [ce_pc.get(ct, 0)  for ct in all_ct]

    n = len(all_ct)
    fig, ax = plt.subplots(figsize=(10, max(4, n * 0.32)))
    y = np.arange(n)
    ax.barh(y - 0.18, ce_vals,  height=0.35, color='tomato',    label='CE Baseline', alpha=0.85)
    ax.barh(y + 0.18, hce_vals, height=0.35, color='steelblue', label='HCE',         alpha=0.85)
    ax.axvline(0.5, color='gray', ls='--', lw=1)
    ax.axvline(0.8, color='gray', ls=':',  lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels(all_ct, fontsize=7)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Recall')
    ax.set_title(f'Per-class Recall — {organ_name}\n'
                 f'HCE macro recall={hce_metrics["macro_recall"]:.3f}  '
                 f'CE macro recall={ce_metrics["macro_recall"]:.3f}')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    safe = organ_name.replace(' ', '_')
    plt.savefig(os.path.join(OUT_DIR, f'recall_comparison_{safe}.png'), dpi=150, bbox_inches='tight')
    plt.show()

for cfg in ORGAN_CONFIGS:
    name = cfg['name']
    if name in results['hce'] and name in results['ce']:
        plot_recall_comparison(name, results['hce'][name], results['ce'][name])

plot_recall_comparison('COMBINED', results['hce']['COMBINED'], results['ce']['COMBINED'])

## 13. Recall Distribution Histogram
*(Does HCE reduce the tail of poor-recall classes?)*

In [ ]:
hce_recalls = results['hce']['COMBINED']['per_class']['recall'].values
ce_recalls  = results['ce']['COMBINED']['per_class']['recall'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins = np.linspace(0, 1, 21)
axes[0].hist(ce_recalls,  bins=bins, alpha=0.7, color='tomato',    label=f'CE  (n={len(ce_recalls)})')
axes[0].hist(hce_recalls, bins=bins, alpha=0.7, color='steelblue', label=f'HCE (n={len(hce_recalls)})')
axes[0].set(xlabel='Per-class recall', ylabel='# classes',
            title='Recall Distribution (all classes)')
axes[0].axvline(0.5, color='gray', ls='--', lw=1, label='50%')
axes[0].legend()
axes[0].grid(alpha=0.3)

# CDF
for recalls, color, label in [(ce_recalls, 'tomato', 'CE'), (hce_recalls, 'steelblue', 'HCE')]:
    sorted_r = np.sort(recalls)
    cdf = np.arange(1, len(sorted_r)+1) / len(sorted_r)
    axes[1].plot(sorted_r, cdf, color=color, lw=2, label=label)
axes[1].set(xlabel='Recall threshold', ylabel='Fraction of classes',
            title='CDF of Per-class Recall\n(higher = more classes above threshold)')
axes[1].axvline(0.5, color='gray', ls='--', lw=1)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('HCE vs CE — Recall Distribution (COMBINED test set)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'recall_distribution_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'  CE  — mean recall: {ce_recalls.mean():.4f}  |  median: {np.median(ce_recalls):.4f}  |  0% classes: {(ce_recalls==0).sum()}')
print(f'  HCE — mean recall: {hce_recalls.mean():.4f}  |  median: {np.median(hce_recalls):.4f}  |  0% classes: {(hce_recalls==0).sum()}')

## 14. Zero-Shot Inference Comparison
*(Both models on lab validation datasets)*

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts, self.tokenizer, self.max_length = texts, tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx] if self.texts[idx].strip() else '[PAD]',
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0)}

def run_inference(model, texts, label):
    inf_loader = DataLoader(
        InferenceDataset(texts, tokenizer, MAX_SEQ_LEN),
        batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True
    )
    preds, confs = [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(inf_loader, desc=f'    [{label}] infer', leave=False):
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            leaf_logits = logits[:, leaf_indices_t]
            probs = torch.softmax(leaf_logits, dim=1)
            pred_pos = leaf_logits.argmax(dim=1)
            preds.extend(leaf_indices_t[pred_pos].cpu().numpy())
            confs.extend(probs.max(dim=1).values.cpu().numpy())
    return [class_names[i] for i in preds], np.array(confs)

lab_results_hce, lab_results_ce = {}, {}

for lab_cfg in LAB_CONFIGS:
    lab_name  = lab_cfg['name']
    label_col = lab_cfg['label_col']
    print(f'\n  --- {lab_name} ---')

    adata_lab    = sc.read_h5ad(lab_cfg['path'])
    lab_gene_sym = get_gene_symbols(adata_lab)
    true_labels  = adata_lab.obs[label_col].astype(str).values
    X_lab        = adata_lab.X
    texts = [cell_to_text_dense(X_lab[i], lab_gene_sym, TOP_K_GENES)
             for i in tqdm(range(adata_lab.n_obs), desc='    text', leave=False)]

    hce_preds_lab, hce_confs_lab = run_inference(hce_model, texts, 'HCE')
    ce_preds_lab,  ce_confs_lab  = run_inference(ce_model,  texts, 'CE')

    lab_results_hce[lab_name] = pd.DataFrame({'true_label': true_labels,
                                               'pred_label': hce_preds_lab,
                                               'confidence': hce_confs_lab})
    lab_results_ce[lab_name]  = pd.DataFrame({'true_label': true_labels,
                                               'pred_label': ce_preds_lab,
                                               'confidence': ce_confs_lab})

    # Print top predictions side by side
    print(f'  {"True label":<40} {"CE pred":>35}  {"HCE pred":>35}')
    print('  ' + '-' * 112)
    for true_lbl in sorted(np.unique(true_labels)):
        ce_top  = lab_results_ce[lab_name][lab_results_ce[lab_name]['true_label']==true_lbl]['pred_label'].value_counts()
        hce_top = lab_results_hce[lab_name][lab_results_hce[lab_name]['true_label']==true_lbl]['pred_label'].value_counts()
        ce_s  = f'{ce_top.index[0]}  ({ce_top.iloc[0]/ce_top.sum()*100:.0f}%)' if len(ce_top) else 'N/A'
        hce_s = f'{hce_top.index[0]}  ({hce_top.iloc[0]/hce_top.sum()*100:.0f}%)' if len(hce_top) else 'N/A'
        print(f'  {true_lbl:<40} {ce_s:>35}  {hce_s:>35}')

print('\n[OK] Zero-shot inference complete')

## 14b. Deeper Biological-Accuracy Analysis
*(Top-K predictions, lineage-match rate, and ancestor recall)*

The zero-shot table above shows only the **mode** prediction per true class. This section pulls three additional views out of the full per-cell prediction DataFrames:

1. **Top-5 predictions per true class** — does HCE's #2/#3 land on a sibling cell type?
2. **Lineage-match rate** — coarse lineage (myeloid / lymphoid / endothelial / mesenchymal / epithelial / glial / neuronal / ...) of predicted vs. true, per zero-shot dataset.
3. **Ancestor recall @ k** — fraction of in-distribution test cells where the predicted leaf and true leaf share an ancestor within ≤ k hops in the ontology. This is the natural metric for HCE.

In [ ]:
# =============================================================================
# Deeper biological-accuracy analysis
#   (1) Top-5 predictions per true class (zero-shot)
#   (2) Lineage-match rate + lineage confusion (zero-shot)
#   (3) Ancestor recall @ k on the in-distribution test set
# =============================================================================

# ---------------------------------------------------------------------------
# Coarse lineage assignment: works on any free-form label by keyword match.
# Order matters: more specific rules first. "microglia" -> Myeloid (yolk-sac
# derived macrophage) even though it sits under glial in some atlases.
# ---------------------------------------------------------------------------
LINEAGE_RULES = [
    ('Myeloid',                 ['macroph','monocyte','microglia','dendritic',' dc ','tam-','tam_','tam ',
                                 'kupffer','neutrophil','mast cell','eosinophil','basophil','myeloid',
                                 'granulocyte','histiocyt','langerhans']),
    ('Lymphoid',                ['t cell','t_cell','cd4','cd8','b cell','b_cell','nk cell','natural killer',
                                 'plasma cell','plasmablast','lymphocyte','lymphoid','innate lymphoid',
                                 ' ilc','follicular helper','regulatory t','gamma-delta','germinal center']),
    ('Erythroid/Megakaryocyte', ['erythro','reticulocyt','megakaryo','platelet']),
    ('Hematopoietic Progenitor',['hematopoietic','hsc',' progenitor','precursor cell']),
    ('Endothelial',             ['endothel','high endothelial venule','ec arterial','ec venous',
                                 'ec general','lymphatic ec','arterial_endothel','venous_endothel',
                                 'capillary_endothel']),
    ('Mesenchymal/Stromal',     ['fibroblast','stellate','pericyte','smooth muscle','smooth_muscle',
                                 'adipocyte','mesenchym','perivascular','meningeal','stroma',
                                 'myofibro','tenocyte','chondrocyte']),
    ('Epithelial',              ['hepatocyt','hepatoblast','cholangio','epithel','enterocyt',
                                 'alveolar type','hepatic cell','goblet','keratino','urothel','tuft']),
    ('Glial',                   ['astrocyt','oligodendro','opc','glia','bergmann','ependymal',
                                 'choroid plexus']),
    ('Neuronal',                ['neuron','interneuron','intratelencephal','hippocampal','thalamic',
                                 'amygdala','rhombic','mammillary','medium spiny',' npc','purkinje',
                                 'granule cell','msn','gabaergic']),
    ('Tumor (glioma-like)',     ['ac-like','mes-like','npc-like','opc-like']),
]

def assign_lineage(label):
    if label is None or (isinstance(label, float) and pd.isna(label)):
        return 'Other'
    s = ' ' + str(label).lower().strip() + ' '
    for lineage, kws in LINEAGE_RULES:
        for kw in kws:
            if kw in s:
                return lineage
    return 'Other'


# ---------------------------------------------------------------------------
# (1) Top-5 predictions per true class, per lab dataset
# ---------------------------------------------------------------------------
print('=' * 100)
print('  (1) TOP-5 PREDICTIONS PER TRUE CLASS  (zero-shot)')
print('=' * 100)
for lab_name in lab_results_hce.keys():
    print(f'\n--- {lab_name} ---')
    df_hce, df_ce = lab_results_hce[lab_name], lab_results_ce[lab_name]
    for true_lbl in sorted(df_hce['true_label'].unique()):
        n = int((df_hce['true_label'] == true_lbl).sum())
        ce_top  = df_ce [df_ce ['true_label']==true_lbl]['pred_label'].value_counts(normalize=True).head(5)
        hce_top = df_hce[df_hce['true_label']==true_lbl]['pred_label'].value_counts(normalize=True).head(5)
        print(f'\n  {true_lbl}  (n={n})')
        print(f'    CE  : ' + '  |  '.join([f'{p}: {v*100:.0f}%' for p, v in ce_top.items()]))
        print(f'    HCE : ' + '  |  '.join([f'{p}: {v*100:.0f}%' for p, v in hce_top.items()]))


# ---------------------------------------------------------------------------
# (2) Lineage-match rate + confusion matrix
# ---------------------------------------------------------------------------
print('\n' + '=' * 100)
print('  (2) LINEAGE-MATCH RATE  (fraction of cells where predicted lineage == true lineage)')
print('=' * 100)
print(f'\n  {"Dataset":<26}{"CE":>10}{"HCE":>10}{"Delta":>10}')
print('  ' + '-' * 56)
lineage_summary = {}
for lab_name in lab_results_hce.keys():
    df_hce, df_ce = lab_results_hce[lab_name], lab_results_ce[lab_name]
    true_lin = df_hce['true_label'].map(assign_lineage)
    ce_lin   = df_ce ['pred_label'].map(assign_lineage)
    hce_lin  = df_hce['pred_label'].map(assign_lineage)
    ce_rate  = float((ce_lin  == true_lin).mean())
    hce_rate = float((hce_lin == true_lin).mean())
    lineage_summary[lab_name] = (ce_rate, hce_rate)
    print(f'  {lab_name:<26}{ce_rate*100:>9.1f}%{hce_rate*100:>9.1f}%{(hce_rate-ce_rate)*100:>+9.2f}')

best_delta_ds = max(lineage_summary, key=lambda k: lineage_summary[k][1] - lineage_summary[k][0])
print(f'\n  Lineage confusion (rows = true lineage, cols = predicted lineage, % per row)')
print(f'  Showing dataset with largest HCE improvement: {best_delta_ds}')
df_h, df_c = lab_results_hce[best_delta_ds], lab_results_ce[best_delta_ds]
tl = df_h['true_label'].map(assign_lineage)
for label, preds_series in [('CE',  df_c['pred_label'].map(assign_lineage)),
                            ('HCE', df_h['pred_label'].map(assign_lineage))]:
    cm = pd.crosstab(tl, preds_series, normalize='index') * 100
    print(f'\n  [{label}] lineage confusion (% per true lineage):')
    print(cm.round(0).astype(int).to_string())


# ---------------------------------------------------------------------------
# (3) Ancestor recall @ k on the IN-DISTRIBUTION TEST SET
#     (both true and predicted labels live in combined_ontology -> clean math)
# ---------------------------------------------------------------------------
def ancestor_set(node, ontology, max_depth=30):
    out, cur, depth = set(), ontology.get(node), 0
    while cur is not None and depth < max_depth and cur not in out:
        out.add(cur); cur = ontology.get(cur); depth += 1
    return out

def lca_hops_from_pred(true_name, pred_name, ontology, max_k=30):
    """Hops climbed from pred until reaching a node in {true} U ancestors(true).
       Returns 0 if pred == true OR pred is an ancestor of true.
       Returns None if no shared ancestor exists."""
    if true_name == pred_name:
        return 0
    true_or_anc = {true_name} | ancestor_set(true_name, ontology, max_k)
    cur, k = pred_name, 0
    while cur is not None and k < max_k:
        if cur in true_or_anc:
            return k
        cur = ontology.get(cur); k += 1
    return None

def ancestor_recall_table(preds, trues):
    cache, depths = {}, []
    for t, p in zip(trues, preds):
        key = (int(t), int(p))
        if key not in cache:
            cache[key] = lca_hops_from_pred(class_names[t], class_names[p], combined_ontology)
        depths.append(cache[key])
    arr = np.array([d if d is not None else 999 for d in depths])
    return {
        'leaf-exact (k=0)':  float((arr == 0).mean()),
        'within k<=1':       float((arr <= 1).mean()),
        'within k<=2':       float((arr <= 2).mean()),
        'within k<=3':       float((arr <= 3).mean()),
        'any shared anc':    float((arr <= 30).mean()),
    }

print('\n' + '=' * 100)
print('  (3) ANCESTOR-RECALL @ k  (in-distribution test set)')
print('  Fraction of cells where pred-leaf shares an ancestor with true-leaf within <=k hops.')
print('  k=0 means pred is either the true leaf or an ancestor of it.')
print('=' * 100)

ce_tbl  = ancestor_recall_table(ce_preds,  ce_trues)
hce_tbl = ancestor_recall_table(hce_preds, hce_trues)
print(f'\n  {"Metric":<22}{"CE":>10}{"HCE":>10}{"Delta":>10}')
print('  ' + '-' * 52)
for k in ce_tbl:
    d = hce_tbl[k] - ce_tbl[k]
    print(f'  {k:<22}{ce_tbl[k]*100:>9.2f}%{hce_tbl[k]*100:>9.2f}%{d*100:>+9.2f}')

print('\n  Per-organ ancestor-recall @ k<=2:')
print(f'  {"Organ":<20}{"CE":>10}{"HCE":>10}{"Delta":>10}')
print('  ' + '-' * 50)
test_organ_np = np.asarray(test_organ)
for cfg in ORGAN_CONFIGS:
    mask = (test_organ_np == cfg['id'])
    if mask.sum() == 0:
        continue
    ce_o  = ancestor_recall_table(ce_preds[mask],  ce_trues[mask])
    hce_o = ancestor_recall_table(hce_preds[mask], hce_trues[mask])
    d = hce_o['within k<=2'] - ce_o['within k<=2']
    print(f'  {cfg["name"]:<20}{ce_o["within k<=2"]*100:>9.2f}%{hce_o["within k<=2"]*100:>9.2f}%{d*100:>+9.2f}')

print('\n[OK] Deep biological-accuracy analysis complete')


## 15. Final Summary

In [ ]:
print('=' * 80)
print('  HCE vs CE BASELINE — FINAL SUMMARY')
print('=' * 80)

print('\nTRAINING')
print(f'  CE  best val accuracy : {ce_best_acc:.4f}')
print(f'  HCE best val accuracy : {hce_best_acc:.4f}')
print(f'  Delta (HCE - CE)      : {hce_best_acc - ce_best_acc:+.4f}')

print('\nCOMBINED TEST SET')
hce_c = results['hce']['COMBINED']
ce_c  = results['ce']['COMBINED']
for metric in ['accuracy', 'macro_recall', 'macro_f1', 'zero_recall_count', 'recall_50_pct', 'recall_80_pct']:
    ce_v, hce_v = ce_c[metric], hce_c[metric]
    delta = hce_v - ce_v
    is_lower_better = metric == 'zero_recall_count'
    fmt = f'{ce_v:.0f}' if metric == 'zero_recall_count' else f'{ce_v:.4f}'
    fmt2 = f'{hce_v:.0f}' if metric == 'zero_recall_count' else f'{hce_v:.4f}'
    fmt3 = f'{delta:+.0f}' if metric == 'zero_recall_count' else f'{delta:+.4f}'
    better_marker = ' (HCE better)' if ((delta > 0) != is_lower_better) or (is_lower_better and delta < 0) else ''
    print(f'  {metric:<25} CE={fmt}  HCE={fmt2}  delta={fmt3}{better_marker}')

print('\nPER-ORGAN MACRO RECALL')
print(f'  {"Organ":<22} {"CE":>8} {"HCE":>8} {"Delta":>10}')
for cfg in ORGAN_CONFIGS:
    name = cfg['name']
    if name not in results['hce'] or name not in results['ce']: continue
    ce_r  = results['ce'][name]['macro_recall']
    hce_r = results['hce'][name]['macro_recall']
    print(f'  {name:<22} {ce_r:>8.4f} {hce_r:>8.4f} {hce_r-ce_r:>+10.4f}')

print(f'\n[OK] Results saved to {OUT_DIR}/')